# Matrix Completion with Truncated SVD

**Topic:** Advanced Linear Algebra &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `M_observed` `(60, 40)`: a rank-5 matrix (plus tiny noise) with **45% of entries missing** (`NaN`).
- `observed_mask`: boolean array, `True` where `M_observed` has a value.
- The held-out true values of the missing entries live inside the evaluation cell.

## Your task

1. Initialize missing entries (column means work). Iterate ~50 times: compute the SVD (`np.linalg.svd`), keep the top **r = 5** singular values, rebuild the matrix, and overwrite only the *missing* positions with the reconstruction (observed positions stay fixed).
2. Set `M_completed` to your final `(60, 40)` matrix.

## Evaluation

RMSE over the missing entries must be below 0.5. Mean-filling scores roughly 4.8, so the low-rank structure has to do the work.


In [ ]:
import numpy as np

rng = np.random.default_rng(231)
n_rows, n_cols, rank = 60, 40, 5

U = rng.normal(0, 1.5, (n_rows, rank))
V = rng.normal(0, 1.5, (rank, n_cols))
M_true = U @ V + rng.normal(0, 0.05, (n_rows, n_cols))

observed_mask = rng.random((n_rows, n_cols)) > 0.45
M_observed = np.where(observed_mask, M_true, np.nan)

print(f"matrix {M_observed.shape}, observed fraction {observed_mask.mean():.2f}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: RMSE on the entries you never saw
# =====================================================================
import numpy as np

if "M_completed" not in dir():
    raise NameError("Define `M_completed` in the solution cell before running this cell.")

_M = np.asarray(M_completed, dtype=float)
if _M.shape != M_true.shape:
    raise ValueError(f"`M_completed` has shape {_M.shape}, expected {M_true.shape}.")

_hidden = ~observed_mask
_rmse = float(np.sqrt(np.mean((_M[_hidden] - M_true[_hidden]) ** 2)))
_col_means = np.nanmean(M_observed, axis=0)
_baseline = float(np.sqrt(np.mean((np.broadcast_to(_col_means, M_true.shape)[_hidden] - M_true[_hidden]) ** 2)))
print("=" * 52)
print(f"Your RMSE on hidden entries : {_rmse:.4f}   (target: < 0.5)")
print(f"Column-mean baseline RMSE   : {_baseline:.4f}")
print("=" * 52)
print("PASSED: low-rank magic!" if _rmse < 0.5 else "NOT PASSED yet: keep only rank 5 and iterate more.")


## Solution

Soft-impute: fill the missing entries with column means, then repeatedly take the SVD, keep the top 5 singular values, and overwrite only the missing positions with the rank-5 reconstruction. Observed entries anchor the iteration.

```python
import numpy as np

M = M_observed.copy()
col_means = np.nanmean(M, axis=0)
M[~observed_mask] = np.broadcast_to(col_means, M.shape)[~observed_mask]

for _ in range(50):
    U, s, Vt = np.linalg.svd(M, full_matrices=False)
    low_rank = (U[:, :5] * s[:5]) @ Vt[:5]
    M[~observed_mask] = low_rank[~observed_mask]   # observed entries stay fixed

M_completed = M
```
